# Employee voice and procedural clarity

This notebook verifies released EWCS 2024 V2 aggregate results. Select **Runtime → Run all**.

The notebook loads the repository directly from GitHub. It verifies 350 table rows, 277 inferential records, seven covariance-derived contrasts and automated tests. It uses aggregate data only. Optional outputs can be enabled in the settings cell.


In [ ]:
from pathlib import Path
from importlib.util import find_spec
import sys, os, subprocess, tempfile

# Optional output settings
GENERATE_FIGURES = False
DOWNLOAD_EXPORT = False

workspace = Path(tempfile.mkdtemp(prefix="employee-voice-"))
repository = Path.cwd().resolve()
if repository.name == "notebooks":
    repository = repository.parent
if not (repository / "code/check_aggregates.py").is_file():
    repository = workspace / "repository"
    subprocess.run([
        "git", "clone", "--quiet", "--depth=1",
        "https://github.com/evidenceworks/employee-voice-procedures.git",
        str(repository)
    ], check=True)

required = [
    "code/verify_public_results.py", "results/T7_Model_contrasts.csv",
    "results/response_state_comparisons.csv", "results/voice_item_response_counts.csv"
]
if not all((repository / name).is_file() for name in required):
    raise RuntimeError("The GitHub repository does not contain the current released aggregate data.")

needed = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.13,<2"}
if GENERATE_FIGURES:
    needed.update({"matplotlib": "matplotlib>=3.8,<4", "PIL": "pillow>=10,<13"})
missing = [spec for package, spec in needed.items() if find_spec(package) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Source repository:", repository)
print("Aggregate-only verification; licensed respondent records are not used.")


## Check published values and derived results

In [ ]:
import json

def run_checked(args):
    run = subprocess.run([sys.executable, *args], cwd=repository,
                         text=True, capture_output=True)
    if run.returncode:
        raise RuntimeError(run.stdout + "\n" + run.stderr)
    return run

verification_dir = workspace / "verification"
args = ["code/verify_public_results.py", "--out", str(verification_dir)]
if GENERATE_FIGURES:
    args.append("--figures")
run_checked(args)
run_checked(["-m", "unittest", "discover", "-s", "tests", "-v"])
report = json.loads((verification_dir / "verification.json").read_text())
assert report["status"] == "PASS"
assert report["primary_aggregate_table_rows"] == 350
assert report["inferential_records"] == 277
assert report["covariance_derived_response_comparisons"] == 7
assert report["respondents_in_analytic_domain"] == 6548
print("Aggregate rows:", report["primary_aggregate_table_rows"])
print("Inference records:", report["inferential_records"])
print("Derived comparisons:", report["covariance_derived_response_comparisons"])
print("Optional figures:", "generated" if GENERATE_FIGURES else "skipped")
print("Checks and tests: PASSED")


## Response profiles and main contrasts

In [ ]:
import csv
from IPython.display import Markdown, display

def read_csv(name):
    with (repository / "results" / name).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

profile = read_csv("T1_Procedure_by_voice.csv")
labels = (("neither", "Neither"), ("representation_only", "Representation only"),
          ("meetings_only", "Meetings only"), ("both", "Both"))
profile_rows = []
for code, label in labels:
    cells = {r["outcome"]:r for r in profile if r["group"] == code}
    profile_rows.append({"configuration": label,"n":int(cells["1"]["group_n"]),
       "yes":float(cells["1"]["weighted_pct"]),"no":float(cells["2"]["weighted_pct"]),
       "unknown":float(cells["3"]["weighted_pct"])})
assert sum(r["n"] for r in profile_rows) == 6548
output = "| Configuration | Employees | Clear procedure (%) | Reports no clear procedure (%) | Does not know (%) |\n|:--|--:|--:|--:|--:|\n"
for r in profile_rows:
    output += f"| {r['configuration']} | {r['n']:,} | {r['yes']:.2f} | {r['no']:.2f} | {r['unknown']:.2f} |\n"
display(Markdown(output))

contrasts=read_csv("T7_Model_contrasts.csv")
for c in ("representation_only - neither", "meetings_only - neither", "both - meetings_only", "additive_interaction"):
    r=next(row for row in contrasts if row["model"]=="A_procedure_yes" and row["contrast"]==c)
    print(f"{c.replace('_',' ')}: {float(r['estimate_pp']):+.2f} pp "
          f"[95% CI {float(r['lo95_pp']):.2f}, {float(r['hi95_pp']):.2f}]")
print("\nUncertainty and negative-answer contrast differences:")
for r in read_csv("response_state_comparisons.csv"):
    print(f"{r['configuration_contrast'].replace('_',' ')}: {float(r['estimate_pp']):+.2f} pp")

## Export the aggregate results and figures

In [ ]:
import hashlib, shutil, zipfile

exports = workspace / "export"
exports.mkdir(exist_ok=True)
for name in ("T1_Procedure_by_voice.csv", "T7_Model_contrasts.csv", "BASELINE_Q57.csv",
             "response_state_comparisons.csv", "voice_item_response_counts.csv"):
    shutil.copy2(repository / "results" / name, exports / name)
shutil.copy2(verification_dir / "verification.json", exports / "verification.json")
if GENERATE_FIGURES:
    for name in ("Figure1_Procedure_profiles.png", "FigureS1_Adjusted_contrasts.png",
                 "FigureS2_Paired_domains.png"):
        shutil.copy2(verification_dir / "figures" / name, exports / name)
archive_path = workspace / "employee-voice-results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(exports.iterdir()):
        archive.write(path, arcname=path.name)
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print("Aggregate export:", archive_path)
print("Files:", len(list(exports.iterdir())))
if DOWNLOAD_EXPORT:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print("Open the export archive directly from the path above.")
